# STAT163 · Week 4 · Before the practice: regular expressions and split

In the practice you will pull parts out of text columns in a table you have not seen yet.
Here you learn the tools for that. Work through it after
`01-before-the-lecture-text-in-columns.ipynb` and before the practice session. Plan about
**45 minutes**.

**Before you start**, work through the interactive course
[Regex 101 on RegexLearn](https://regexlearn.com/learn/regex101), step by step. In each step
you type a pattern and see at once what it matches. Here we assume that you have done it.
In the course you meet more pieces than we use, and the table below lists the ones you
need this week.

You will:

- find the characters that mean something in a pattern, and search for them as characters
- write one pattern for two spellings of the same word
- pull a number out of a text value into a column of its own, and read the rows where that
  failed
- split a value at a separator, and find the rows where the separator is part of the value
- replace text before you split, count or compare it

**How to use it.** The cells are of the same three kinds as in the first notebook: Read and
run, Predict, and Try it. When you run one of the cells, you get an error. That is on
purpose, so go through the cells one at a time rather than with "Run all".

## The pieces of a pattern

These are the pieces of a pattern you need this week, in one table. Keep it open during
the practice.

| Piece | Means |
|---|---|
| `\d` | one digit |
| `[A-Z]` | one capital letter |
| `[A-Za-z]` | one letter, capital or small |
| `[- ]` | one of the characters between the brackets: here a hyphen or a space |
| `[\d.]` | one digit or a dot: between square brackets a dot is an ordinary character |
| `.` | any one character |
| `?` | the piece before it, zero times or once |
| `+` | the piece before it, one or more times |
| `{5}` | the piece before it, five times |
| `{1,2}` | the piece before it, once or twice |
| `\b` | the edge of a word: the border between a letter or a digit and anything else, such as a space, a bracket, or the start or the end of the value |
| `( )` | the part to pull out |
| `\.` `\(` `\)` | a dot or a bracket as an ordinary character |

## Load the table

In `names` we keep each product name once.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/stat163-2026t1/week3-pre-lecture/main/data/online_retail_2010_11.csv"
df = pd.read_csv(url)

names = df["Description"].dropna().drop_duplicates()

## Part 1 — Characters with a special meaning

Product names are not only letters, digits and spaces. Which other characters appear in
them, and in how many names? We count each character as plain text, with `regex=False`:

In [2]:
print(len(names))
for character in [".", ",", "/", "(", "+", "&", "-", "'", "?"]:
    print(character, names.str.contains(character, regex=False).sum())

3089
. 28
, 99
/ 167
( 1
+ 50
& 20
- 91
' 29
? 2


Some of these characters mean something in a pattern: `.`, `(`, `+` and `?`. What happens
when you search for one of them without `regex=False`?

In [ ]:
# Predict: how many names contain "." when we leave out regex=False?
names.str.contains(".").sum()

<details>
<summary>Answer</summary>

Every name. In a pattern, `.` stands for any one character, so every name that has at
least one character matches.

</details>

A bracket goes wrong in another way.

In [ ]:
# Predict: what happens when you search for "(" without regex=False?
names.str.contains("(")

<details>
<summary>Answer</summary>

An error. In a pattern, `(` is the start of the part to pull out, and a pattern with a `(`
and no `)` is broken.

</details>

There are two ways to search for the character itself:

In [5]:
print(names.str.contains("(", regex=False).sum())
print(names.str.contains(r"\(").sum())

1
1


With `regex=False` you tell pandas to treat the text as plain text. With the backslash in
`\(` you mark this one bracket as an ordinary character, and you can still use the other
pieces of a pattern around it. Put a backslash before `.`, `(`, `)`, `[`, `]`, `?`, `+`,
`{` or `}` whenever you mean the character itself.

The dot matters most when you look for numbers. A few product names have a decimal number
in them, such as `ESSENTIAL BALM 3.5g TIN IN ENVELOPE`.

In [ ]:
# Predict: which of the two counts is larger, and what do the extra names have in them?
print(names.str.contains(r"\d.\d").sum())
print(names.str.contains(r"\d\.\d").sum())

<details>
<summary>Answer</summary>

The first. With `\d.\d` you ask for a digit, then any character, then a digit, so three
digits in a row match too, and so do a digit, a letter and a digit. With `\d\.\d` you ask
for a digit, a dot and a digit.

</details>

Read the names that match only the first pattern:

In [7]:
names[names.str.contains(r"\d.\d") & ~names.str.contains(r"\d\.\d")].head(8)

461                 BAG 125g SWIRLY MARBLES
712          SET OF 6 3D KIT CARDS FOR KIDS
783                 BAG 500g SWIRLY MARBLES
785                 BAG 250g SWIRLY MARBLES
1269           POLYESTER FILLER PAD 40x40cm
2214                      code mix up 72597
2622           POLYESTER FILLER PAD 45x45cm
2726    PINK AND WHITE CHRISTMAS TREE 120CM
Name: Description, dtype: str

## Part 2 — A word with more than one spelling

When you count products by a word in their name, you count only the spelling you typed.
How many product names have the `RETROSPOT` design? Compare the count with a shorter
search, `RETRO`:

In [8]:
print(names.str.contains("RETROSPOT").sum())
print(names.str.contains("RETRO").sum())

94
119


More names contain `RETRO`. Take out the `RETROSPOT` names and read the rest:

In [9]:
is_retro = names.str.contains("RETRO")
is_retrospot = names.str.contains("RETROSPOT")
names[is_retro & ~is_retrospot].sort_values()

9275               BIRTHDAY CARD, RETRO SPOT
3455           MAGNETS PACK OF 4 RETRO PHOTO
1308        PACK OF 72 RETRO SPOT CAKE CASES
1625               PAPER BUNTING RETRO SPOTS
1624              PAPER CHAIN KIT RETRO SPOT
56921              PINK RETRO BIG FLOWER BAG
606                  PINK/PURPLE RETRO RADIO
107                     RETRO "TEA FOR ONE" 
1717              RETRO COFFEE MUGS ASSORTED
43699            RETRO LAUNDRY TUB PISTACHIO
587      RETRO LONGBOARD IRONING BOARD COVER
2828                          RETRO MOD TRAY
8936                 RETRO PLASTIC 70'S TRAY
708                 RETRO PLASTIC DAISY TRAY
7160             RETRO PLASTIC ELEPHANT TRAY
8934                RETRO PLASTIC POLKA TRAY
35663     RETRO RED SPOTTY WASHING UP GLOVES
1312          RETRO SPOT  CIGAR BOX MATCHES 
308                RETRO SPOT LARGE MILK JUG
4358           RETRO SPOT SMALL TUBE MATCHES
1328       RETRO SPOT TEA SET CERAMIC 11 PC 
849          SET 12 RETRO WHITE CHALK STICKS
1401     S

Some of them are other retro products: a radio, a tray, coffee mugs. Others have the same
design written as two words, `RETRO SPOT`. With `?` after the space you allow the space
once or not at all, so with one pattern you count both spellings:

In [10]:
names.str.contains(r"RETRO ?SPOT").sum()

np.int64(103)

With square brackets you allow a choice of characters. Suppose you do not know how the
shop writes tea lights, small candles in a metal cup, and you allow three spellings:
`T-LIGHT`, `T LIGHT` and `TLIGHT`. With `[- ]?` you allow a hyphen, a space or nothing
between the two parts.

In [ ]:
# Predict: the second pattern allows three spellings. Which names match the second
# pattern and not the first?
is_tea_light = names.str.contains("T-LIGHT")
is_wide = names.str.contains(r"T[- ]?LIGHT")
names[is_wide & ~is_tea_light]

<details>
<summary>Answer</summary>

None of them is a tea light: two night lights, a nightlight and
`ART LIGHTS,FUNK MONKEY`. The `T` at the end of `NIGHT` and `ART` matches, and the other
two spellings of tea light do not appear in this table at all.

</details>

With `\b` you ask for the edge of a word. Put it in front of the `T`, and the `T` has to
start a word:

In [12]:
print(is_tea_light.sum())
print(names.str.contains(r"\bT[- ]?LIGHT").sum())

81
81


Now the two counts are the same. So before you widen a pattern, check that the other
spellings exist, and read the names you gain.

## Part 3 — Pulling a number out of the text

A text value often holds a number that you would like as a column of its own: a size, a
weight, a count. Many product names have a number in them. What do those numbers mean?
From here on we work with the rows of `df`, so that each line of a receipt gets its own
value. Here are the most common lines whose name has a digit:

In [13]:
df.loc[df["Description"].str.contains(r"\d"), "Description"].value_counts().head(15)

Description
PAPER CHAIN KIT 50'S CHRISTMAS         345
REGENCY CAKESTAND 3 TIER               315
60 CAKE CASES VINTAGE CHRISTMAS        224
PACK OF 72 RETROSPOT CAKE CASES        210
SET OF 20 VINTAGE CHRISTMAS NAPKINS    210
RETROSPOT TEA SET CERAMIC 11 PC        199
FELTCRAFT 6 FLOWER FRIENDS             196
6 RIBBONS RUSTIC CHARM                 163
3 HEARTS HANGING DECORATION RUSTIC     161
SET OF 3 NOTEBOOKS IN PARCEL           158
PACK OF 6 BIRDY GIFT TAGS              152
SET/5 RED RETROSPOT LID GLASS BOWLS    142
CHRISTMAS LIGHTS 10 REINDEER           133
SET 7 BABUSHKA NESTING BOXES           130
60 TEATIME FAIRY CAKE CASES            130
Name: count, dtype: int64

Read the list. Which numbers are the size of a pack, and which describe the product itself?

<details>
<summary>Answer</summary>

The size of a pack: `PACK OF 72`, `SET OF 20`, `SET/5`, `SET 7`, `60 CAKE CASES`,
`6 RIBBONS`. Part of the product: `3 TIER`, `3 HEARTS`, `10 REINDEER`, and `50'S`, for the
1950s.

</details>

The pack size is written in several ways. We start with the most common one: `OF`, a
space, and a number.

The part of a pattern inside `( )` is called a **group**. With `extract` you pull out the
part of the text that matches the group. Try it on three names first:

In [14]:
example = pd.Series([
    "PACK OF 72 RETROSPOT CAKE CASES",
    "SET OF 20 VINTAGE CHRISTMAS NAPKINS",
    "REGENCY CAKESTAND 3 TIER",
])
example.str.extract(r"OF (\d+)", expand=False)

0     72
1     20
2    NaN
dtype: str

You get the digits after `OF `, the part inside the group. The third name does not match,
so its value is missing, and you get no error. With `expand=False` you get one column.
Without it, you get a table with one column for each group in the pattern.

Now the whole column. We put the result beside the name, so that you can check it:

In [15]:
pack_size = df["Description"].str.extract(r"OF (\d+)", expand=False)

print(len(df))
print(pack_size.notna().sum())
df[["Description"]].assign(pack_size=pack_size).dropna().head()

78015
4700


,Description,pack_size
21,BOX OF 6 ASSORTED COLOUR TEASPOONS,6
30,PACK OF 6 SWEETIE GIFT BOXES,6
34,SET OF 6 STRAWBERRY CHOPSTICKS,6
100,BABUSHKA LIGHTS STRING OF 10,10
104,SET OF 6 T-LIGHTS TOADSTOOLS,6


Not every match is a pack size: in `BABUSHKA LIGHTS STRING OF 10`, 10 is the number of
lights on one string. Read the rows that did not match too. Look at the `SET` rows, with
the result beside each name:

In [16]:
steps = df[["Description"]].assign(pack_size=pack_size)
steps[steps["Description"].str.contains("SET")].drop_duplicates("Description").head(6)

,Description,pack_size
18,SET OF THREE VINTAGE GIFT WRAPS,NaN
33,CHILDS BREAKFAST SET CIRCUS PARADE,NaN
34,SET OF 6 STRAWBERRY CHOPSTICKS,6
45,AIRLINE BAG VINTAGE JET SET RED,NaN
50,SET 7 BABUSHKA NESTING BOXES,NaN
53,SET/6 PURPLE BUTTERFLY T-LIGHTS,NaN


`SET OF 6` matched. `SET OF THREE` has the number in words, `SET 7` has no `OF`, and
`SET/6` has a slash. The slash spelling has a shape we can describe: a `/`, then digits.
We pull those numbers out with a second pattern, and fill the missing pack sizes from it.
Again we look at every step side by side:

In [17]:
slash_size = df["Description"].str.extract(r"/(\d+)", expand=False)
combined = pack_size.fillna(slash_size)

steps = df[["Description"]].assign(of=pack_size, slash=slash_size, pack_size=combined)
steps[steps["Description"].str.contains("SET")].drop_duplicates("Description").head(6)

,Description,of,slash,pack_size
18,SET OF THREE VINTAGE GIFT WRAPS,NaN,NaN,NaN
33,CHILDS BREAKFAST SET CIRCUS PARADE,NaN,NaN,NaN
34,SET OF 6 STRAWBERRY CHOPSTICKS,6,NaN,6
45,AIRLINE BAG VINTAGE JET SET RED,NaN,NaN,NaN
50,SET 7 BABUSHKA NESTING BOXES,NaN,NaN,NaN
53,SET/6 PURPLE BUTTERFLY T-LIGHTS,NaN,6,6


With `fillna` and a second column, you fill each missing value from the same row of the
second column: in `pack_size` you get the value of `of` where there is one, and the value
of `slash` elsewhere. `SET OF THREE` and `SET 7` are still missing. You could write a pattern
for them too, or decide that they are few enough to leave out.

What you pull out of text is text. To turn it into numbers, use `pd.to_numeric` and then
`Int64`:

In [18]:
print(combined.dtype)
df["pack_size"] = pd.to_numeric(combined).astype("Int64")
print(df["pack_size"].dtype)

str
Int64


### Pulling out several parts

A pattern can have more than one group. A product code such as `85123A` has two parts:
five digits for the product and a letter for its variant. With `{1,2}` we allow one letter
or two, and we leave out `expand=False`:

In [19]:
code_parts = df["StockCode"].str.extract(r"(\d{5})([A-Za-z]{1,2})")
code_parts.head()

,0,1
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


The first codes in the table have no letter, so they do not match, and both columns are
missing. The columns are named `0` and `1`. We give them names and add them to the table:

In [20]:
df[["code_number", "variant"]] = code_parts
df.loc[df["variant"].notna(), ["StockCode", "code_number", "variant"]].head(3)

,StockCode,code_number,variant
9,85123A,85123,A
20,85114A,85114,A
31,84569B,84569,B


Each part is now a column you can use in a groupby. Which products come in the most
variants?

In [21]:
df.groupby("code_number")["variant"].nunique().sort_values(ascending=False).head(3)

code_number
90214    24
84596    12
85049    11
Name: variant, dtype: int64

Some of these variants are the same letter, once capital and once small, as in `84596b`
and `84596B`. Text is case-sensitive: in a pattern and in a comparison, `b` and `B` are two
different values. With `.str.upper()` you turn every letter into a capital, and then you
count again:

In [22]:
(
    df
    .assign(variant=df["variant"].str.upper())
    .groupby("code_number")["variant"]
    .nunique()
    .sort_values(ascending=False)
    .head(3)
)

code_number
90214    24
85049     8
84596     7
Name: variant, dtype: int64

### A group next to special characters

For the next examples we go back to `names`, each product name once. You can mix all the
pieces in one pattern. To pull out the text between two brackets, put a group inside `\(`
and `\)`:

In [23]:
names.str.extract(r"\(([A-Za-z ]+)\)", expand=False).dropna()

8101    POINTED EDGE
Name: Description, dtype: str

Inside the group we allow only letters and spaces, `[A-Za-z ]+`, so the group ends at the
first `)`. With `.+` in its place, the group would run to the last `)` in the value, and a
value with two brackets, such as `(American) (British)`, would give `American) (British`.

With `[\d.]+` you allow one or more characters that are each a digit or a dot, so you pull
out a number whether it has a decimal part or not. Here are the weights in grams from the
product names:

In [24]:
names.str.extract(r"([\d.]+)g\b", expand=False).dropna()

461    125
620    3.5
783    500
785    250
Name: Description, dtype: str

Some names give a size in centimetres, written `CM` in some names and `cm` in others. Some
give two sizes, as in `POLYESTER FILLER PAD 45x30cm`. In the next cell we pull out the
number before `CM`, in capitals:

In [25]:
# Try it: change the pattern so that you get both numbers of a size written like 45x30cm,
# one column for each, with [\d.]+ for each number
names.str.extract(r"([\d.]+)CM", expand=False).dropna()

59        15
1659      40
1781      15
1970      60
2726     120
2727      60
5692      30
11590     65
12104     60
12105     60
15366     60
17712     30
23619     50
25060    120
35218     16
57302     30
57475     30
Name: Description, dtype: str

Check your result against the names. `THE KING GIFT BAG 25x24x12cm` has three sizes, and
with two groups you get only the last two of them.

## Part 4 — Splitting at a separator

Sometimes one value holds several pieces with the same character between them, like a
short list written in one cell. You can cut the value at that character. In some product
names a comma stands between the kind of product and its design. Start with one name, in
plain Python:

In [26]:
"WRAP, BILLBOARD FONTS DESIGN".split(",")

['WRAP', ' BILLBOARD FONTS DESIGN']

You get a list of two pieces, and the second piece starts with a space. With `.str.split`
you do the same for every value of a column:

In [27]:
with_comma = names[names.str.contains(",")]
pieces = with_comma.str.split(",")
pieces.head()

139    [WALL MIRROR ,  RECT DIAMANTE,  PINK/]
341      [WALL MIRROR ,  DIAMANTE OVAL SHAPE]
414      [SET 3 RETROSPOT TEA, COFFEE, SUGAR]
453             [ELEPHANT,  BIRTHDAY CARD,  ]
507          [CHRISTMAS GARLAND STARS, TREES]
Name: Description, dtype: object

The pieces of each list are printed with a comma and a space between them, so
`[SET 3 RETROSPOT TEA, COFFEE, SUGAR]` is three pieces. With `.str[0]` you take the first
piece of each list, as with `[0]` on a Python list. With `.str[1]` you take the second
piece, and with `.str[-1]` the last one. Look at the first pieces before and after you
remove the spaces around them with `.str.strip()`:

In [28]:
first = pieces.str[0]
print(first.head(3).tolist())
print(first.str.strip().head(3).tolist())

['WALL MIRROR ', 'WALL MIRROR ', 'SET 3 RETROSPOT TEA']
['WALL MIRROR', 'WALL MIRROR', 'SET 3 RETROSPOT TEA']


Now count the kinds of product that the first pieces give:

In [29]:
first.str.strip().value_counts().head()

Description
FRENCH BLUE METAL DOOR SIGN    11
NUMBER TILE COTTAGE GARDEN     11
KEY FOB                         4
HOOK                            4
CAKE TIN                        4
Name: count, dtype: int64

Before you trust the first piece, count how many pieces each name has. With `.str.len()`
you get the length of each list:

In [30]:
pieces.str.len().value_counts()

Description
2    78
3    18
4     3
Name: count, dtype: int64

Read the names with more than two pieces:

In [31]:
with_comma[pieces.str.len() > 2].head(10)

139       WALL MIRROR , RECT DIAMANTE, PINK/
414         SET 3 RETROSPOT TEA,COFFEE,SUGAR
453                ELEPHANT, BIRTHDAY CARD, 
2655            HOOK, 1 HANGER ,MAGIC GARDEN
4687           NURSERY A,B,C PAINTED LETTERS
4930             BLACK TEA,COFFEE,SUGAR JARS
8855     SET 3 RED SPOT TIN TEA,COFFEE,SUGAR
10761        BISCUIT TIN, RED,IVORY, VINTAGE
12423               DECOUPAGE,GREETING CARD,
15745    HOOK, 5 HANGER ,MAGIC TOADSTOOL RED
Name: Description, dtype: str

In some of them the commas still separate parts of the description, as in
`HOOK, 1 HANGER ,MAGIC GARDEN`. In `SET 3 RETROSPOT TEA,COFFEE,SUGAR` and
`NURSERY A,B,C PAINTED LETTERS` the commas sit inside one part, a list of things, and the
first piece is no longer the kind of product. The same character is a separator in one
name and part of the value in another, and you find out which only by reading the rows.

A count of pieces is useful only when you know how many to expect. When a value can hold any
number of pieces, read the pieces themselves, not only the first one.

Now try another separator. Many names have a `/`, as in `PINK/WHITE CHRISTMAS TREE 60CM`.

In [32]:
# Try it: change "," to "/" in both places, and read the first pieces
names[names.str.contains(",")].str.split(",").str[0].str.strip().value_counts().head()

Description
FRENCH BLUE METAL DOOR SIGN    11
NUMBER TILE COTTAGE GARDEN     11
KEY FOB                         4
HOOK                            4
CAKE TIN                        4
Name: count, dtype: int64

The two most common first pieces are `SET` and `S`, from names such as `SET/6` and `S/4`,
where the `/` belongs to the size of a set. From those first pieces you learn nothing
about the product.

## Part 5 — Replacing text

When a comma belongs to the value, replace it before you split. With
`.str.replace(old, new)` you replace every `old` in a value with `new`. Here we write the
list of tea, coffee and sugar with slashes instead of commas, and count the pieces again:

In [33]:
fixed = with_comma.str.replace("TEA,COFFEE,SUGAR", "TEA/COFFEE/SUGAR")

print(fixed[fixed.str.contains("TEA/")].tolist())
fixed.str.split(",").str.len().value_counts()

['SET 3 RETROSPOT TEA/COFFEE/SUGAR', 'BLACK TEA/COFFEE/SUGAR JARS', 'SET 3 RED SPOT TIN TEA/COFFEE/SUGAR', 'WHITE TEA/COFFEE/SUGAR JARS']


Description
2    78
3    14
1     4
4     3
Name: count, dtype: int64

The tea, coffee and sugar names now have no comma left, so each of them is one piece: the
whole name.

You need the same tool when one value is written in two ways. The two most common first
pieces in Part 4 came from door signs. Read the names of the door signs with their codes:

In [34]:
signs = df.loc[
    df["Description"].str.startswith("FRENCH BLUE METAL DOOR SIGN"),
    ["StockCode", "Description"],
]
signs.drop_duplicates().sort_values("StockCode").head(6)

,StockCode,Description
3644,22676,"FRENCH BLUE METAL DOOR SIGN, 1"
17933,22676,FRENCH BLUE METAL DOOR SIGN 1
17927,22677,FRENCH BLUE METAL DOOR SIGN 2
24401,22677,"FRENCH BLUE METAL DOOR SIGN, 2"
15548,22678,FRENCH BLUE METAL DOOR SIGN 3
24400,22678,"FRENCH BLUE METAL DOOR SIGN, 3"


Each code has two names, one with a comma before the number and one without. Replace the
comma with nothing, and count the different names again:

In [35]:
sign_names = signs["Description"].drop_duplicates()

print(sign_names.nunique())
print(sign_names.str.replace(",", "").nunique())

22
11


Now every sign has one name. Before you count or compare names, check whether the same
thing is written in more than one way.

With `contains`, the text you pass is read as a pattern unless you add `regex=False`. With
`str.replace` it is the other way round: the text is plain text unless you add
`regex=True`. Some names have two spaces where one belongs, as in
`COLUMBIAN  CUBE CANDLE`. The pattern ` +`, a space and `+`, means one or more spaces in a
row.

In [ ]:
# Predict: how many names still have two spaces in a row after this replacement?
one_space = names.str.replace(r" +", " ")
one_space.str.contains("  ", regex=False).sum()

<details>
<summary>Answer</summary>

All of them. Without `regex=True`, ` +` is plain text, a space and a plus sign, and no
name has it, so nothing was replaced.

</details>

With `regex=True` the pattern works. We also remove the spaces at both ends with
`.str.strip()`, and count the different names before and after:

In [37]:
cleaned = names.str.replace(r" +", " ", regex=True).str.strip()

print(names.nunique())
print(cleaned.nunique())

3089
3083


A few names were the same name, written with different spaces. In the list you see each
pair as it was written:

In [38]:
sorted(names[cleaned.duplicated(keep=False)].tolist())

['BATHROOM METAL SIGN',
 'BATHROOM METAL SIGN ',
 'COLUMBIAN  CUBE CANDLE ',
 'COLUMBIAN CUBE CANDLE',
 'HEART T-LIGHT HOLDER',
 'HEART T-LIGHT HOLDER ',
 'PINK JEWELLED PHOTO FRAME',
 'PINK JEWELLED PHOTO FRAME ',
 'ROSE DU SUD CUSHION COVER',
 'ROSE DU SUD CUSHION COVER ',
 'SET OF 4 FAIRY CAKE PLACEMATS',
 'SET OF 4 FAIRY CAKE PLACEMATS ']

## What to check before you trust a pattern or a split

- Does my pattern have a character with a special meaning that I meant as an ordinary
  character?
- Is the thing I am looking for written in more than one way, and do the other spellings
  exist in this table?
- After an extraction, what is in the rows that did not match?
- After a split, how many pieces does each value have, is the separator ever part of a
  value, and what is in the pieces after the first?
- Before I split, count or compare, is the same value written in more than one way, with a
  comma or with extra spaces, and did I replace it first?